# धडा 09 - मेटाकॉग्निशन डिझाइन पॅटर्न


## सेटअप

हे नोटबुक Microsoft Agent Framework वापरून Metacognition डिझाइन पॅटर्न दाखवते.

**पूर्वअट:**
- Azure OpenAI तैनाती जी पर्यावरण चलांद्वारे कॉन्फिगर केलेली असावी
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मेटाकॉग्निशन म्हणजे काय?

मेटाकॉग्निशन म्हणजे **विचाराबद्दल विचार करणे**. AI एजंट्सच्या संदर्भात, याचा अर्थ असे एजंट तयार करणे जे करू शकतात:

- **स्वप्रतिबिंबित होणे** त्यांच्या स्वतःच्या आउटपुट्स आणि तर्क प्रक्रियेवर
- **त्रुटी ओळखणे** आणि मूकपणे अयशस्वी होण्याऐवजी योग्य प्रकारे पुनर्प्राप्त होणे
- **मूल्यांकन करणे** की त्यांच्या प्रतिसाद पूर्ण आणि उपयुक्त आहेत का
- **अनुकूलन करणे** त्यांच्या धोरणाला जेव्हा प्रारंभिक पद्धत काम करत नाही (उदा., बॅकअप सिस्टीमकडे परत जाणे)

एक मेटाकॉग्निशन एजंट फक्त प्रश्नांची उत्तरे देत नाही — तो स्वतःच्या कामगिरीवर लक्ष ठेवतो आणि तत्काळ समायोजित करतो.


## प्राथमिक आणि बॅकअप साधने

एक सामान्य मेटाकॉग्निशन पॅटर्न म्हणजे **बॅकअप धोरण**. एजंट प्रथम एका प्राथमिक साधनाचा प्रयत्न करतो; ते अपयशी झाल्यास (उदा., 404 त्रुटी), एजंट त्या अपयशाची ओळख करून घेतो आणि पारदर्शकपणे बॅकअप साधनावर स्विच करतो.

हे वास्तविक जगातील प्रणालींशी साम्य दाखवते जिथे प्राथमिक सेवा उपलब्ध नसू शकतात आणि एजंटांनी पर्यायी मार्ग निवडण्यापूर्वी स्वतःच समस्येचे निदान करणे आवश्यक असते.

खाली आम्ही दोन फ्लाइट शोध साधने परिभाषित करतो:
- **प्राथमिक** — पॅरिस, टोकियो आणि बार्सिलोना समाविष्ट करते
- **बॅकअप** — बर्लिन, सिडनी आणि न्यू यॉर्क सिटी समाविष्ट करते


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटी पुनर्प्राप्तीसह स्व-परावलोकन करणारा एजंट

खालील एजंटला प्रथम प्राथमिक विमानचालन प्रणाली वापरून पाहण्याचे, अपयश ओळखण्याचे, आणि पारदर्शकपणे बॅकअप प्रणालीकडे परत जाण्याचे निर्देश दिले आहेत. प्रत्येक प्रतिसादानंतर ते संक्षेपाने स्वतःचे मूल्यांकन करते की ते वापरकर्त्याच्या प्रश्नाचे पूर्णपणे उत्तर दिले का.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## स्व-मूल्यांकन नमुना

मेटाकॉग्निशनचा आणखी एक पैलू म्हणजे **स्व-मूल्यांकन**: एक वेगळा एजंट (किंवा दुसऱ्या फेरीत तोच एजंट) प्रतिसादाची पूर्णता, अचूकता आणि उपयुक्तता तपासतो.

खाली आपण एक `ResponseEvaluator` एजंट तयार करतो जो प्रवास-एजंटच्या प्रतिसादांना तीन परिमाणांवर गुण देतो.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून **मेटाकॉग्निटिव्ह एजंट** कसे तयार करायचे हे शिकले:

- **स्व-चिंतन**: असे एजंट जे त्यांच्या स्वतःच्या विचारप्रक्रियेवर लक्ष ठेवतात आणि काय झाले ते पारदर्शकपणे माहिती देतात.
- **फॉलबॅक्ससह त्रुटी पुनर्प्राप्ती**: हा एक प्राथमिक + बॅकअप टूल पॅटर्न आहे जिथे एजंट अपयश ओळखतो (उदा., 404 त्रुटी) आणि आपोआप पर्यायी स्रोताचा प्रयत्न करतो.
- **स्व-मूल्यांकन**: एक स्वतंत्र मूल्यांकन करणारा एजंट जो प्रतिसादांचे पूर्णत्व, अचूकता आणि उपयुक्ततेसाठी गुणवारणी करतो.

हे पॅटर्न एजंट्सना अधिक मजबूत, पारदर्शक आणि विश्वासार्ह बनवतात — उत्पादनात तैनात करण्यासाठी आवश्यक गुणधर्म.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केलेला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अपूर्णता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत म्हणून मान्य केला जावा. महत्वाच्या माहितीच्या बाबतीत व्यावसायिक मानवी अनुवाद शिफारस करण्यात येतो. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थविषयक परिणामांसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
